# Patron creacional: Prototype

Prototype es un patrón de diseño creacional que nos permite copiar objetos existentes sin que el código dependa de sus clases.

![patron_prototype](img/diagrama_prototype.png)

## Cuando Usarlo?
* Utiliza el patrón Prototype cuando tu código no deba depender de las clases concretas de objetos que necesites copiar.
* Utiliza el patrón cuando quieras reducir la cantidad de subclases que solo se diferencian en la forma en que inicializan sus respectivos objetos. Puede ser que alguien haya creado estas subclases para poder crear objetos con una configuración específica.

## Pros y contras
#### Pros
* Puedes clonar objetos sin acoplarlos a sus clases concretas.
* Puedes evitar un código de inicialización repetido clonando prototipos prefabricados.
* Puedes crear objetos complejos con más facilidad.
* Obtienes una alternativa a la herencia al tratar con preajustes de configuración para objetos complejos.

#### Contras
* Clonar objetos complejos con referencias circulares puede resultar complicado.

## Codigo

#### Importaciones

In [1]:
import copy

In [2]:
class SelfReferencingEntity:
    def __init__(self):
        self.parent = None

    def set_parent(self, parent):
        self.parent = parent

In [3]:
class SomeComponent:
    """
    Python provee su propia interfaz de Prototype mediante las 
    funciones 'copy.copy' y 'copy.deepcopy'. Y cualquier clase que quiera
    implementar implementaciones personalizadas tiene que sobreescribir
    los metodos '__copy__' and '__deepcopy__'.
    """

    def __init__(self, some_int, some_list_of_objects, some_circular_ref):
        self.some_int = some_int
        self.some_list_of_objects = some_list_of_objects
        self.some_circular_ref = some_circular_ref

    def __copy__(self):
        """
        Crea una copia superficial. Se llamará a este método cada vez que alguien 
        llame a `copy.copy` con este objeto y el valor devuelto se devuelva como 
        la nueva copia superficial.
        """

        # First, let's create copies of the nested objects.
        some_list_of_objects = copy.copy(self.some_list_of_objects)
        some_circular_ref = copy.copy(self.some_circular_ref)

        # Then, let's clone the object itself, using the prepared clones of the
        # nested objects.
        new = self.__class__(
            self.some_int, some_list_of_objects, some_circular_ref
        )
        new.__dict__.update(self.__dict__)

        return new

    def __deepcopy__(self, memo=None):
        """
        Crea una copia profunda. Se llamará a este método cada vez que 
        alguien llame a `copy.deepcopy` con este objeto y el valor devuelto 
        se devuelva como la nueva copia profunda.
        
        ¿Cuál es el uso del argumento `memo`? Memo es el diccionario que 
        utiliza la libreria `deepcopy` para evitar infinitas copias recursivas 
        en instancias de referencias circulares. Páselo a todas las llamadas 
        `deepcopy` que realice en la implementación `__deepcopy__` para evitar 
        recursiones infinitas.
        """
        if memo is None:
            memo = {}

        # Primero, creemos copias de los objetos anidados.
        some_list_of_objects = copy.deepcopy(self.some_list_of_objects, memo)
        some_circular_ref = copy.deepcopy(self.some_circular_ref, memo)

        # Luego, clonemos el objeto mismo, usando los clones preparados de los 
        # objetos anidados.
        new = self.__class__(
            self.some_int, some_list_of_objects, some_circular_ref
        )
        new.__dict__ = copy.deepcopy(self.__dict__, memo)

        return new

In [5]:
if __name__ == "__main__":

    list_of_objects = [1, {1, 2, 3}, [1, 2, 3]]
    circular_ref = SelfReferencingEntity()
    component = SomeComponent(23, list_of_objects, circular_ref)
    circular_ref.set_parent(component)

    shallow_copied_component = copy.copy(component)

    # Cambiemos la lista en componente copiado_superficial y veamos si 
    # cambia en component.
    shallow_copied_component.some_list_of_objects.append("another object")
    if component.some_list_of_objects[-1] == "another object":
        print(
            "Adding elements to `shallow_copied_component`'s "
            "some_list_of_objects adds it to `component`'s "
            "some_list_of_objects."
        )
    else:
        print(
            "Adding elements to `shallow_copied_component`'s "
            "some_list_of_objects doesn't add it to `component`'s "
            "some_list_of_objects."
        )

    # Cambiemos el conjunto en la lista de objetos.
    component.some_list_of_objects[1].add(4)
    if 4 in shallow_copied_component.some_list_of_objects[1]:
        print(
            "Changing objects in the `component`'s some_list_of_objects "
            "changes that object in `shallow_copied_component`'s "
            "some_list_of_objects."
        )
    else:
        print(
            "Changing objects in the `component`'s some_list_of_objects "
            "doesn't change that object in `shallow_copied_component`'s "
            "some_list_of_objects."
        )

    deep_copied_component = copy.deepcopy(component)

    # Cambiemos la lista en deep_copied_component y veamos si cambia en component.
    deep_copied_component.some_list_of_objects.append("one more object")
    if component.some_list_of_objects[-1] == "one more object":
        print(
            "Adding elements to `deep_copied_component`'s "
            "some_list_of_objects adds it to `component`'s "
            "some_list_of_objects."
        )
    else:
        print(
            "Adding elements to `deep_copied_component`'s "
            "some_list_of_objects doesn't add it to `component`'s "
            "some_list_of_objects."
        )

    # Cambiemos el conjunto en la lista de objetos.
    component.some_list_of_objects[1].add(10)
    if 10 in deep_copied_component.some_list_of_objects[1]:
        print(
            "Changing objects in the `component`'s some_list_of_objects "
            "changes that object in `deep_copied_component`'s "
            "some_list_of_objects."
        )
    else:
        print(
            "Changing objects in the `component`'s some_list_of_objects "
            "doesn't change that object in `deep_copied_component`'s "
            "some_list_of_objects."
        )

    print(
        f"id(deep_copied_component.some_circular_ref.parent): "
        f"{id(deep_copied_component.some_circular_ref.parent)}"
    )
    print(
        f"id(deep_copied_component.some_circular_ref.parent.some_circular_ref.parent): "
        f"{id(deep_copied_component.some_circular_ref.parent.some_circular_ref.parent)}"
    )
    print(
        "^^ This shows that deepcopied objects contain same reference, they "
        "are not cloned repeatedly."
    )

Adding elements to `shallow_copied_component`'s some_list_of_objects adds it to `component`'s some_list_of_objects.
Changing objects in the `component`'s some_list_of_objects changes that object in `shallow_copied_component`'s some_list_of_objects.
Adding elements to `deep_copied_component`'s some_list_of_objects doesn't add it to `component`'s some_list_of_objects.
Changing objects in the `component`'s some_list_of_objects doesn't change that object in `deep_copied_component`'s some_list_of_objects.
id(deep_copied_component.some_circular_ref.parent): 140388720020928
id(deep_copied_component.some_circular_ref.parent.some_circular_ref.parent): 140388720020928
^^ This shows that deepcopied objects contain same reference, they are not cloned repeatedly.
